# M5.S2 — From basic MPI/OpenMP to scalable applications
**IE University · 2026–27 BCSAI · HPC · official Session 22**

The official syllabus calls for an **integrated application perspective**: how MPI and OpenMP combine communication, threading, synchronization and data decomposition. This is **not** an advanced MPI syntax course.

**Learning outcomes:** decompose a grid across ranks; explain threads versus processes; exchange boundary rows; identify synchronization and load-balance limits; compare correct serial-equivalent, MPI-only, OpenMP-only and hybrid execution.

**Problem → Prediction → Code → Run → Result → Explanation → Student experiment → Takeaway.**

The notebook displays **no invented MPI timings**. Actual SciTech runs remain **NOT RUN** unless the Slurm job executes and returns authentic logs.

**Course progression:** Session 21 Hybrid computing patterns → **Session 22 Scalable MPI/OpenMP applications** → Session 23 Containers, portability and reproducibility. Main syllabus reading: *Foundations of High-Performance Computing*, Chapters 7 and 8.

## 0. Get ready in IE JupyterLab (10 min)
1. Open `https://jupyter.iesci.tech:8085/` and authenticate with your university account.
2. **File → Open from URL…** paste the notebook raw GitHub link in the adjacent README; choose **Python 3** and save your own copy.
3. Jupyter may run on a login/shared host. MPI measurements belong in an **allocated Slurm CPU job**, not uncontrolled `mpirun` on the login node.
4. Check `sinfo`, your active MPI compiler module and `cluster/README.md`. This demonstration stays on **one node** because earlier M2.S3 work found multi-node MPI configuration issues.
5. Editing inputs changes results only after a **new** job is run. `RUN_CLUSTER=False` is safe by default. Optional Matplotlib for observed results only; no package installation.

In [ ]:
from pathlib import Path
import os, json, hashlib, shutil, subprocess, time, math, re
LAB=Path.cwd()/"m5s2_scalable_lab"
LAB.mkdir(parents=True,exist_ok=True)
RUN_CLUSTER=False             # Opt in only after checking actual SciTech resources.
PARTITION="cpu"               # Confirm with sinfo.
MPI_MODULE="foss/2023b"      # Confirm with module avail; use "" when preloaded.
GRID_N=96                    # Change only one: e.g. 48, 96, 192.
STEPS=40                     # e.g. 20 or 80.
print("Personal lab:",LAB.resolve())
print("Computer:",os.uname().nodename,"| Slurm allocation:",os.environ.get("SLURM_JOB_ID","none"))
for name in ["gcc","mpicc","sbatch","srun","squeue"]:
    print(name,shutil.which(name) or "not found on this host")
print("RUN_CLUSTER:",RUN_CLUSTER,"(no Slurm submission unless True)")


## 1. [PREDICT] A real scientific problem: heat spreading across a plate (10 min)
A square hotspot (100 units) is surrounded by cooler cells (zero). One Jacobi step sets each interior cell to the average of its **four neighbors from the previous step**.

**Before running:** Does the hotspot spread? Can a value on the plate's outer boundary change? What could go wrong if threads overwrite the old grid while other threads still read it?

The small Python reference below performs **real local computation**, not an MPI benchmark.

In [ ]:
def plate(n):
    return [[100.0 if n//3<=y<(2*n)//3 and n//3<=x<(2*n)//3 else 0.0 for x in range(n)] for y in range(n)]
def jacobi(u):
    n=len(u);v=[[0.0]*n for _ in range(n)]
    for y in range(1,n-1):
        for x in range(1,n-1):
            v[y][x]=0.25*(u[y-1][x]+u[y+1][x]+u[y][x-1]+u[y][x+1])
    return v
a=plate(12);start=sum(map(sum,a))
for _ in range(6):
    a=jacobi(a)
assert all(v==0 for v in a[0]+a[-1])
assert all(row[0]==row[-1]==0 for row in a)
assert 0<=sum(map(sum,a))<=start
print("PASS local Python reference: start_sum=",start,"after_6_steps=",round(sum(map(sum,a)),3))
print("MPI/Slurm is NOT RUN by this cell.")


## 2. [DISCUSS] Who owns each cell? MPI ranks and OpenMP threads (10 min)
**Data decomposition:** divide the **interior rows** into contiguous bands. Each MPI rank owns a band and temporarily stores two **halo** rows for neighbors' boundary data.

**MPI:** processes have distinct address spaces, even when they run on the same physical machine; ranks exchange the halo rows.

**OpenMP:** threads inside one rank share that rank's arrays and compute different owned rows in parallel.

**[PREDICT]** With 4 ranks, where are messages needed? With 2 ranks × 2 threads, where is each rank's synchronization? Which work can happen without a message?

In [ ]:
def partition(n,p):
    inside=n-2
    assert 1<=p<=inside
    base,rem=divmod(inside,p)
    return [(1+r*base+min(r,rem),base+(r<rem)) for r in range(p)]
rows=partition(14,4)
covered=[y for first,count in rows for y in range(first,first+count)]
assert covered==list(range(1,13))
assert max(count for _,count in rows)-min(count for _,count in rows)<=1
for rank,(first,count) in enumerate(rows):
    print(f"MPI rank {rank}: own rows {first}..{first+count-1}; halo {first-1} and {first+count}")
print("PASS: all interior rows assigned once; balanced within one row.")


## 3. One C program for all four configurations (10 min)
The next cell writes a **real C + MPI + OpenMP heat solver**, not pseudocode.

- `MPI_Init_thread(MPI_THREAD_FUNNELED)` and ranks select a row band.
- `MPI_Sendrecv` safely exchanges the neighboring halo rows at each step.
- `#pragma omp parallel for` updates the local rows with shared-memory CPU threads.
- `MPI_Gatherv` collects the final plate. Rank 0 independently recomputes a **serial reference** and reports numerical error, checksum and elapsed time.
- `model_halo_payload_bytes` is a **calculated application payload**, **not** measured network throughput.

**[PREDICT]** Do different configurations need to produce identical output? Yes; their performance may differ.

In [ ]:
HEAT_SOURCE="#include <mpi.h>\n#include <omp.h>\n#include <math.h>\n#include <stdio.h>\n#include <stdlib.h>\n#define A(a,y,x,n) (a[(size_t)(y)*(n)+(x)])\nstatic int hotspot(int y,int x,int n){return y>=n/3&&y<2*n/3&&x>=n/3&&x<2*n/3;}\nint main(int argc,char**argv){\n  int level=0;MPI_Init_thread(&argc,&argv,MPI_THREAD_FUNNELED,&level);\n  int rank=0,size=1;MPI_Comm_rank(MPI_COMM_WORLD,&rank);MPI_Comm_size(MPI_COMM_WORLD,&size);\n  if(argc!=4||level<MPI_THREAD_FUNNELED){if(rank==0)fprintf(stderr,\"Usage: heat N STEPS THREADS\\n\");MPI_Finalize();return 2;}\n  int n=atoi(argv[1]),steps=atoi(argv[2]),threads=atoi(argv[3]);\n  if(n<16||n>512||steps<1||steps>400||threads<1||threads>8||size>n-2){\n    if(rank==0)fprintf(stderr,\"Invalid grid, steps, threads or rank count\\n\");\n    MPI_Finalize();return 2;\n  }\n  omp_set_dynamic(0);omp_set_num_threads(threads);\n  int base=(n-2)/size,rest=(n-2)%size;\n  int rows=base+(rank<rest),start=1+rank*base+(rank<rest?rank:rest);\n  double *u=calloc((size_t)(rows+2)*n,sizeof(double));\n  double *v=calloc((size_t)(rows+2)*n,sizeof(double));\n  if(!u||!v){MPI_Abort(MPI_COMM_WORLD,3);return 3;}\n  for(int y=1;y<=rows;y++)for(int x=1;x<n-1;x++)\n    A(u,y,x,n)=hotspot(start+y-1,x,n)?100.0:0.0;\n  int top=rank?rank-1:MPI_PROC_NULL,bottom=rank<size-1?rank+1:MPI_PROC_NULL;\n  MPI_Barrier(MPI_COMM_WORLD);double t0=MPI_Wtime();\n  for(int step=0;step<steps;step++){\n    MPI_Sendrecv(&A(u,1,0,n),n,MPI_DOUBLE,top,51,\n                 &A(u,rows+1,0,n),n,MPI_DOUBLE,bottom,51,\n                 MPI_COMM_WORLD,MPI_STATUS_IGNORE);\n    MPI_Sendrecv(&A(u,rows,0,n),n,MPI_DOUBLE,bottom,52,\n                 &A(u,0,0,n),n,MPI_DOUBLE,top,52,\n                 MPI_COMM_WORLD,MPI_STATUS_IGNORE);\n    #pragma omp parallel for schedule(static)\n    for(int y=1;y<=rows;y++)for(int x=1;x<n-1;x++)\n      A(v,y,x,n)=.25*(A(u,y-1,x,n)+A(u,y+1,x,n)+A(u,y,x-1,n)+A(u,y,x+1,n));\n    double *tmp=u;u=v;v=tmp;\n  }\n  MPI_Barrier(MPI_COMM_WORLD);double elapsed=MPI_Wtime()-t0;\n  int *counts=NULL,*disp=NULL;double *full=NULL;\n  if(rank==0){\n    counts=calloc((size_t)size,sizeof(int));disp=calloc((size_t)size,sizeof(int));\n    full=calloc((size_t)n*n,sizeof(double));\n    if(!counts||!disp||!full){MPI_Abort(MPI_COMM_WORLD,4);return 4;}\n    for(int r=0;r<size;r++){\n      int owned=base+(r<rest),first=1+r*base+(r<rest?r:rest);\n      counts[r]=owned*n;disp[r]=first*n;\n    }\n  }\n  MPI_Gatherv(&A(u,1,0,n),rows*n,MPI_DOUBLE,full,counts,disp,MPI_DOUBLE,0,MPI_COMM_WORLD);\n  long long bytes=(long long)steps*n*sizeof(double)*((top!=MPI_PROC_NULL)+(bottom!=MPI_PROC_NULL));\n  long long total=0;\n  MPI_Reduce(&bytes,&total,1,MPI_LONG_LONG_INT,MPI_SUM,0,MPI_COMM_WORLD);\n  int fail=0;\n  if(rank==0){\n    double *ref=calloc((size_t)n*n,sizeof(double)),*next=calloc((size_t)n*n,sizeof(double));\n    if(!ref||!next){MPI_Abort(MPI_COMM_WORLD,5);return 5;}\n    for(int y=0;y<n;y++)for(int x=0;x<n;x++)A(ref,y,x,n)=hotspot(y,x,n)?100.0:0.0;\n    for(int step=0;step<steps;step++){\n      for(int y=1;y<n-1;y++)for(int x=1;x<n-1;x++)\n        A(next,y,x,n)=.25*(A(ref,y-1,x,n)+A(ref,y+1,x,n)+A(ref,y,x-1,n)+A(ref,y,x+1,n));\n      double *tmp=ref;ref=next;next=tmp;\n    }\n    double err=0,checksum=0;\n    for(int y=0;y<n;y++)for(int x=0;x<n;x++){\n      double dif=fabs(A(full,y,x,n)-A(ref,y,x,n));\n      if(dif>err)err=dif;checksum+=A(full,y,x,n);\n    }\n    fail=err>1e-9||!isfinite(checksum);\n    printf(\"RESULT status=%s n=%d steps=%d ranks=%d threads=%d seconds=%.8f max_error=%.3e checksum=%.6f model_halo_payload_bytes=%lld\\n\",\n           fail?\"FAIL\":\"PASS\",n,steps,size,threads,elapsed,err,checksum,total);\n    fflush(stdout);\n    free(ref);free(next);free(counts);free(disp);free(full);\n  }\n  MPI_Bcast(&fail,1,MPI_INT,0,MPI_COMM_WORLD);\n  free(u);free(v);MPI_Finalize();return fail?1:0;\n}"
C_PATH=LAB/"heat_mpi_omp.c"
C_PATH.write_text(HEAT_SOURCE+"\n",encoding="utf-8")
print("Prepared C source:",C_PATH)
for line in HEAT_SOURCE.splitlines():
    if any(k in line for k in ("MPI_Sendrecv(","#pragma omp","MPI_Gatherv(","RESULT status=")):
        print(line.strip()[:170])


## 4. Controlled SciTech job: serial, MPI-only, OpenMP-only, hybrid (5 min setup)
One short **single-node** CPU allocation, up to **four CPU cores**, six-minute limit, 1 GiB. Every run computes the **same heat-diffusion problem**.

| Mode | MPI ranks | OpenMP threads per rank | Main question |
|---|---:|---:|---|
| S | 1 | 1 | What is the reference performance? |
| M | 4 | 1 | Which halo messages are exchanged? |
| O | 1 | 4 | How can threads share the local arrays? |
| H | 2 | 2 | What changes with fewer rank boundaries? |
| B | 4 | 1 | What happens for a smaller plate? |

**Notes:** 4-rank / 1-rank steps run sequentially within the same batch job. Actual Slurm task binding is cluster-dependent: if a step is rejected, report **FAIL/SKIPPED**. Do not oversubscribe or claim inter-node scaling.

In [ ]:
import shlex
assert 24 <= GRID_N <= 256, 'Choose a safe classroom grid size between 24 and 256'
assert 1 <= STEPS <= 120, 'Keep total steps within the classroom time budget'
def script_text():
    mod=(f"if command -v module >/dev/null 2>&1; then module load {shlex.quote(MPI_MODULE)} || true; fi"
         if MPI_MODULE.strip() else ": # use active MPI compiler environment")
    parts=[
      "#!/usr/bin/env bash","#SBATCH --job-name=m5s2_scalable","#SBATCH --nodes=1",
      "#SBATCH --ntasks=4","#SBATCH --cpus-per-task=1","#SBATCH --time=00:06:00",
      "#SBATCH --mem=1G","set -uo pipefail",
      "cd "+shlex.quote(str(LAB.resolve())),mod,
      'if ! command -v mpicc >/dev/null 2>&1; then echo "JOB_STATUS=SKIPPED reason=no_mpicc"; exit 10; fi',
      'if ! command -v srun >/dev/null 2>&1; then echo "JOB_STATUS=SKIPPED reason=no_srun"; exit 10; fi',
      'if ! mpicc -std=c11 -O2 -Wall -Wextra -fopenmp heat_mpi_omp.c -lm -o heat_mpi_omp; then echo "JOB_STATUS=FAIL reason=compile"; exit 2; fi',
      'echo "COMPILE status=PASS"','failed=0',
      'run_mode(){ label="$1"; np="$2"; nt="$3"; n="$4"; steps="$5";',
      '  echo "CASE_BEGIN name=$label ranks=$np threads=$nt n=$n steps=$steps"',
      '  if OMP_NUM_THREADS="$nt" OMP_DYNAMIC=FALSE srun --exact -N1 -n"$np" -c"$nt" --cpu-bind=cores ./heat_mpi_omp "$n" "$steps" "$nt"; then',
      '    echo "CASE_END name=$label status=COMPLETED"',
      '  else echo "CASE_END name=$label status=FAIL"; failed=1; fi;',
      '}',
      f"run_mode S 1 1 {GRID_N} {STEPS}",f"run_mode M 4 1 {GRID_N} {STEPS}",
      f"run_mode O 1 4 {GRID_N} {STEPS}",f"run_mode H 2 2 {GRID_N} {STEPS}",
      f"run_mode B 4 1 {max(16,GRID_N//3)} {STEPS}",
      'echo "JOB_STATUS_DONE failed=$failed"','exit "$failed"']
    return "\n".join(parts)+"\n"
SBATCH=LAB/"m5s2_scalable.sbatch"
SBATCH.write_text(script_text(),encoding="utf-8")
fingerprint=hashlib.sha256((HEAT_SOURCE+script_text()).encode()).hexdigest()[:12]
print("Script:",SBATCH,"fingerprint:",fingerprint)
print(script_text())
print("NOT RUN: generating the script does not submit a job.")


## 5. Opt-in submission and duplicate-job guard
**[PREDICT]** Which mode might be fastest and why? A model is not proof: correctness first, actual timing second.

Default **RUN_CLUSTER=False**. When enabled on SciTech with `sbatch`, submit at most once per source/configuration fingerprint. A previous queued/running job blocks new parameter configurations. If the compiler is unavailable or the scheduler rejects the job, report the actual failure rather than simulating success.

In [ ]:
state_file=LAB/"m5s2_slurm_state.json"
job_id=None
if not RUN_CLUSTER:
    print("NOT RUN: Slurm execution not requested.")
elif not shutil.which("sbatch"):
    print("SKIPPED: no sbatch; use approved SciTech compute resources.")
else:
    previous=json.loads(state_file.read_text()) if state_file.exists() else {}
    if previous.get("fingerprint")==fingerprint and previous.get("job_id"):
        job_id=str(previous["job_id"])
        print("REUSED previous job",job_id,"for unchanged inputs.")
    else:
        prior=str(previous.get("job_id",""));active=False
        if prior.isdigit() and shutil.which("squeue"):
            proc=subprocess.run(["squeue","-h","-j",prior],capture_output=True,text=True,timeout=20)
            active=(proc.returncode==0 and bool(proc.stdout.strip()))
        if active:
            print("SKIPPED: previous job",prior,"still active. Wait before resubmitting.")
        else:
            cmd=["sbatch","--parsable","-p",PARTITION,
                 "--output",str(LAB/f"m5s2_{fingerprint}_%j.out"),
                 "--error",str(LAB/f"m5s2_{fingerprint}_%j.err"),str(SBATCH)]
            env=os.environ.copy()
            for key in ("SLURM_MEM_PER_CPU","SLURM_MEM_PER_GPU","SLURM_MEM_PER_NODE"):
                env.pop(key,None)
            res=subprocess.run(cmd,capture_output=True,text=True,timeout=30,env=env)
            if res.returncode:
                print("FAIL: sbatch error:",res.stderr[:900])
            else:
                candidate=res.stdout.strip().split(";")[0]
                if not candidate.isdigit():
                    print("FAIL: unexpected sbatch response",res.stdout[:300])
                else:
                    job_id=candidate
                    state_file.write_text(json.dumps({"fingerprint":fingerprint,"job_id":job_id},indent=2))
                    print("SUBMITTED:",job_id)
if job_id: print("Track with: squeue -j",job_id)


## 6. Read the actual job logs and classify evidence
If a job is queued, rerun the following cell after it completes. Do **not** confuse a source file or an incomplete output with a passed experiment.

Each `CASE_BEGIN/CASE_END` pair marks a genuine run. A mode counts as **PASS** only if its numerical correctness check returns PASS and the case completes. If a step is rejected or has no result, it is **FAIL/NOT RUN**.

In [ ]:
def read_job_output(wait=30):
    if not job_id:
        print("NOT RUN: no allocated SciTech job.");return ""
    out=LAB/f"m5s2_{fingerprint}_{job_id}.out"
    err=LAB/f"m5s2_{fingerprint}_{job_id}.err"
    end=time.monotonic()+wait
    while time.monotonic()<end:
        if out.exists() and "JOB_STATUS_DONE" in out.read_text(errors="replace"): break
        time.sleep(2)
    if not out.exists(): print("NOT RUN / QUEUED: output not ready.");return ""
    content=out.read_text(errors="replace")
    print("REAL JOB OUTPUT:",out,"\n",content[-12000:])
    if err.exists() and err.stat().st_size:
        print("REAL STDERR:",err.read_text(errors="replace")[-2000:])
    if "JOB_STATUS_DONE" not in content: print("INCOMPLETE: the job has not confirmed completion.")
    return content
JOB_OUTPUT=read_job_output()


In [ ]:
def classify(text):
    results={m:{"status":"NOT RUN"} for m in "SMOHB"}
    if not text: return results
    mode=None
    for line in text.splitlines():
        if line.startswith("CASE_BEGIN "):
            found=re.search(r"name=([SMOHB])\b",line);mode=found.group(1) if found else None
        elif line.startswith("RESULT status=") and mode:
            stat=re.search(r"status=(PASS|FAIL)",line)
            sec=re.search(r"seconds=([\d.]+)",line)
            error=re.search(r"max_error=([\deE+.-]+)",line)
            bytes_=re.search(r"model_halo_payload_bytes=(\d+)",line)
            if stat: results[mode]["status"]=stat.group(1)
            if sec: results[mode]["seconds"]=float(sec.group(1))
            if error: results[mode]["max_error"]=float(error.group(1))
            if bytes_: results[mode]["model_payload_bytes"]=int(bytes_.group(1))
        elif line.startswith("CASE_END "):
            found=re.search(r"name=([SMOHB]).*status=(\w+)",line)
            if found and found.group(2)!="COMPLETED":results[found.group(1)]["status"]="FAIL"
            mode=None
    return results
RESULTS=classify(JOB_OUTPUT)
for key,title in [("S","Serial-equivalent"),("M","MPI-only 4×1"),("O","OpenMP-only 1×4"),("H","Hybrid 2×2"),("B","Small grid MPI")]:
    print(f"{key} {title:22} {RESULTS[key]}")


## 7. [DEMO] MPI: rank ownership and halo messages (10 min)
**Problem:** Distribute one scientific plate over 4 processes with separate memory.

**[PREDICT]** Which row data must cross rank boundaries at every timestep?

**[RUN]** Compare S (1×1) with M (4×1). `MPI_Sendrecv` exchanges **halo rows**, not the entire plate. Check numerical error before time.

**[ACTIVITY]** Change one setting, `GRID_N` 96→192, re-run with a fresh fingerprint and compare the *actual* correctness and runtime.

**[TAKEAWAY]** Decomposing across MPI ranks creates communication responsibilities.

In [ ]:
print("Actual run A:",RESULTS["S"])
print("Actual run B:",RESULTS["M"])
if RESULTS["S"]["status"]!="PASS" or RESULTS["M"]["status"]!="PASS":
    print("MEASUREMENT NOT VERIFIED. Predict now, compare only after real execution.")

## 8. [DEMO] OpenMP: parallelize a rank's local loop (10 min)
**Problem:** Threads in one MPI process share the process's arrays.

**[PREDICT]** Does `#pragma omp parallel for` automatically spread work across physical nodes? Could overwriting old values create a dependency bug?

**[RUN]** Compare S (1×1) and O (1×4). Inspect the stencil update directive, barriers and the two-grid swap.

**[ACTIVITY]** Explain what breaks if the program reads and writes the same plate in-place.

**[TAKEAWAY]** OpenMP shares local memory: parallel work still needs correct data dependencies.

In [ ]:
print("Actual run A:",RESULTS["S"])
print("Actual run B:",RESULTS["O"])
if RESULTS["S"]["status"]!="PASS" or RESULTS["O"]["status"]!="PASS":
    print("MEASUREMENT NOT VERIFIED. Predict now, compare only after real execution.")

## 9. [DISCUSS/DEMO] Hybrid decomposition: 4×1 or 2×2? (10 min)
**Problem:** Four allocated CPU cores could execute as four MPI processes, or two MPI processes with two threads each.

**[PREDICT]** Which configuration may have fewer rank boundaries? Why might fewer messages **not** imply faster execution?

**[RUN]** Compare M (4×1) to H (2×2); discuss thread placement, OpenMP startup cost and imbalance. The payload counter is a *model*, not a network measurement.

**[ACTIVITY]** Draw a two-rank row decomposition and two thread workers per rank.

**[TAKEAWAY]** Combine MPI and OpenMP according to data ownership, locality and observed bottlenecks.

In [ ]:
print("Actual run A:",RESULTS["M"])
print("Actual run B:",RESULTS["H"])
if RESULTS["M"]["status"]!="PASS" or RESULTS["H"]["status"]!="PASS":
    print("MEASUREMENT NOT VERIFIED. Predict now, compare only after real execution.")

## 10. [PREDICT/DEMO] Where scalability breaks (10 min)
**Problem:** Small problems can perform too little work to offset process/thread costs.

**[PREDICT]** Is 4-rank MPI faster on a small 32×32 plate? Why could a larger plate behave differently?

**[RUN]** B is the same MPI-only kernel with a smaller plate; compare to M only when both complete. Single-node timings are illustrative and affected by noise.

**[ACTIVITY]** Change `STEPS` 40→80, regenerate and re-run. Do not invent timings if SciTech is unavailable.

**[TAKEAWAY]** More parallelism does not automatically mean more performance.

In [ ]:
print("Actual run A:",RESULTS["M"])
print("Actual run B:",RESULTS["B"])
if RESULTS["M"]["status"]!="PASS" or RESULTS["B"]["status"]!="PASS":
    print("MEASUREMENT NOT VERIFIED. Predict now, compare only after real execution.")

## 11. Actual-data-only comparison chart
The plot below is drawn only from successful real SciTech runs. A missing run shows **NOT RUN**, not an invented bar or an assumed speedup.

In [ ]:
actual=[(name,r) for name,r in RESULTS.items() if r["status"]=="PASS" and "seconds" in r]
if not actual:
    print("NOT RUN: no SciTech measurements; chart intentionally omitted.")
else:
    for name,r in actual:
        print(f"Mode {name}: {r['seconds']:.7f} s, error={r.get('max_error')}, model bytes={r.get('model_payload_bytes')}")
    try:
        import matplotlib.pyplot as plt
        fig,ax=plt.subplots(figsize=(6,3))
        ax.bar([n for n,_ in actual],[r["seconds"] for _,r in actual])
        ax.set(xlabel="Execution mode",ylabel="Actual elapsed seconds",
               title="Single-node measured heat simulation: one run per mode")
        plt.show()
    except ImportError:
        print("SKIPPED: Matplotlib not installed. Actual numerical results are displayed above.")


## 12. [TAKEAWAY] Four essentials and an exit question (5 min)
1. **Decompose:** MPI ranks own separate rows and exchange halos.
2. **Parallelize locally:** OpenMP threads update distinct local rows in shared memory.
3. **Synchronize correctly:** each new timestep requires completed communication and consistent old/new data.
4. **Measure before optimizing:** correctness, communication overhead, thread startup, data sizes and load balance determine results.

**Exit question:** For four CPU cores, would you choose **4 MPI ranks × 1 OpenMP thread** or **2 ranks × 2 threads**? Justify with data ownership, expected communication, a correctness check and one possible bottleneck.

**Conceptual extensions only:** non-blocking MPI, one-sided communication and persistent requests may help some sophisticated applications, but are not required for this Bachelor-level practical.

**Next:** Session 23, containers and reproducible HPC workflows.